In [8]:
from datetime import datetime

def get_current_time():
    """Returns the current time as string"""
    return datetime.now().strftime('%H:%M:%S')

In [10]:
# from langchainupdated.1-langchainintro import messages
import aisuite as ai



In [15]:
messages = [{"role": "user", "content": "What is the time now?"}]

In [ ]:
client = ai.Client()

response = client.chat.completions.create(
    model = 'ollama:llama3.2:3b',
    messages = messages,
    tools = get_current_time(),
    max_turns = 5
)

I'm not currently able to share the time.


In [ ]:
import aisuite as ai

client = ai.Client()

messages = [{"role": "user", "content": "What is the time now?"}]

# NOTE: For actual tool execution, you have to define the tool schema.
# For now, let's get the basic chat completion working.
response = client.chat.completions.create(
    model = 'ollama:llama3.2:3b',
    messages = messages
)

print(response.choices[0].message.content)


In [19]:
import aisuite as ai
from datetime import datetime
import json

# 1. Define your python function
def get_current_time():
    """Returns the current time as string"""
    return datetime.now().strftime('%H:%M:%S')

# 2. Define the tool schema (OpenAI format)
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Returns the current time as a string.",
            # If your function took arguments, you'd define them here in "parameters"
        }
    }
]

client = ai.Client()

messages = [{"role": "user", "content": "What is the time now?"}]

# 3. First call: We ask the model and provide the tools
print("Asking model...")
response = client.chat.completions.create(
    model='ollama:llama3.2:3b',
    messages=messages,
    tools=tools
)

response_message = response.choices[0].message
messages.append(response_message) # Add the model's response to the conversation history

# 4. Check if the model decided to call a tool
if response_message.tool_calls:
    for tool_call in response_message.tool_calls:
        if tool_call.function.name == "get_current_time":
            print("Model requested to call: get_current_time()")
            
            # Execute our actual Python function
            time_result = get_current_time()
            print(f"Tool returned: {time_result}")
            
            # Add the result of the tool to the conversation
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": time_result
            })
            
    # 5. Second call: Send the tool's result back to the model so it can answer the user
    print("Sending tool result back to model...")
    final_response = client.chat.completions.create(
        model='ollama:llama3.2:3b',
        messages=messages,
        tools=tools
    )
    
    print("\n--- Final Answer ---")
    print(final_response.choices[0].message.content)
else:
    # If the model didn't need to call a tool, it just answers directly
    print("\n--- Final Answer ---")
    print(response_message.content)


Asking model...

--- Final Answer ---
I'm not currently able to share the time.


In [20]:
client

In [23]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
import os
from datetime import datetime

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# 1. Define the tool using the @tool decorator (much simpler than JSON schemas!)
@tool
def get_current_time() -> str:
    """Returns the current time as a string."""
    return datetime.now().strftime('%H:%M:%S')

# 2. Initialize your Gemini model
model = init_chat_model("google_genai:gemini-2.5-flash-lite")

# 3. Bind the tool to the model
model_with_tools = model.bind_tools([get_current_time])

# 4. Ask the model
print("Asking Gemini...")
response = model_with_tools.invoke("What is the time now?")

# 5. Check if it decided to call a tool
if response.tool_calls:
    for tool_call in response.tool_calls:
        print(f"Model requested to call: {tool_call['name']}")
        
        # Execute the tool
        if tool_call['name'] == "get_current_time":
            time_result = get_current_time.invoke(tool_call['args'])
            print(f"Tool returned: {time_result}")
            
            # Send the result back to the model
            # Langchain handles the message history for tool results using ToolMessage
            from langchain_core.messages import ToolMessage
            
            tool_msg = ToolMessage(content=time_result, tool_call_id=tool_call['id'])
            
            # We send the original question, the model's tool request, and our tool result
            final_response = model_with_tools.invoke([
                ("user", "What is the time now?"),
                response,
                tool_msg
            ])
            
            print("\n--- Final Answer ---")
            print(final_response.content)
else:
    print("\n--- Final Answer ---")
    print(response.content)


Asking Gemini...
Model requested to call: get_current_time
Tool returned: 06:05:01

--- Final Answer ---
The current time is 06:05:01.
